# Aula 09 — Correlação e Regressão Linear Simples

## Objetivos
1. Calcular e interpretar correlações de **Pearson** e **Spearman**.
2. Ajustar uma **regressão linear simples** por mínimos quadrados.
3. Avaliar o modelo: $R^2$, resíduos, p-valores dos coeficientes.
4. Distinguir correlação de causalidade.

---

## 1. Correlação

### Pearson (linear)
$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}$$

- Mede associação **linear**.
- $r \in [-1, 1]$.
- Sensível a outliers e a relações não-lineares.

### Spearman (de postos)
- Aplica Pearson nos **rankings** dos dados.
- Mede qualquer relação **monotônica**.
- Robusto a outliers.

### Regra de bolso
| $|r|$ | Força |
|-------|-------|
| 0,0 – 0,2 | Muito fraca |
| 0,2 – 0,4 | Fraca |
| 0,4 – 0,6 | Moderada |
| 0,6 – 0,8 | Forte |
| 0,8 – 1,0 | Muito forte |

## 2. Regressão Linear Simples

Modelo:
$$Y_i = \beta_0 + \beta_1 X_i + \varepsilon_i, \quad \varepsilon_i \sim N(0,\sigma^2)$$

Estimadores por **Mínimos Quadrados Ordinários (MQO)**:
$$\hat\beta_1 = \frac{\sum (x_i-\bar x)(y_i-\bar y)}{\sum (x_i-\bar x)^2}$$
$$\hat\beta_0 = \bar y - \hat\beta_1 \bar x$$

### $R^2$ (coeficiente de determinação)
$$R^2 = 1 - \frac{\text{SQR}}{\text{SQT}} = \frac{\text{SQE}}{\text{SQT}}$$

Proporção da variância de $Y$ **explicada** pelo modelo.

### Pressupostos (a verificar com resíduos)
1. **Linearidade** — relação real é linear.
2. **Independência** dos resíduos.
3. **Homocedasticidade** — variância constante.
4. **Normalidade** dos resíduos.

## 3. Correlação ≠ Causalidade

Vendas de sorvete e afogamentos correlacionam — ambos sobem no verão. A relação é
**espúria**: a temperatura causa as duas. Sempre questione mecanismos antes de inferir causalidade.

In [ ]:
# === SETUP PARA GOOGLE COLAB ===
# Esta célula garante que o notebook funcione tanto em ambiente local
# quanto em quem clonar o repositório direto no Colab.

import sys, os, subprocess

# 1) Instala dependências (silencioso se já existirem)
pkgs = ["numpy", "pandas", "matplotlib", "seaborn", "scipy", "requests", "plotly", "statsmodels"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# 2) Se estivermos no Colab e o repo ainda não foi clonado, clona.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/220719/curso-estatistica-python.git"  # ajuste após criar o repo

if IN_COLAB and not os.path.exists("/content/curso-estatistica-python"):
    subprocess.run(["git", "clone", REPO_URL, "/content/curso-estatistica-python"], check=False)

# 3) Adiciona o diretório utils ao PYTHONPATH para importar o módulo SIDRA
BASE = "/content/curso-estatistica-python" if IN_COLAB else ".."
if BASE not in sys.path:
    sys.path.append(BASE)

# 4) Pasta de gráficos (cria se não existir)
GRAFICOS_DIR = os.path.join(BASE, "graficos")
os.makedirs(GRAFICOS_DIR, exist_ok=True)

print("✓ Ambiente pronto. Pasta de gráficos:", GRAFICOS_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import os

sns.set_theme(style="whitegrid")

from utils.sidra_api import obter_pib_per_capita_uf, obter_populacao_uf

## 4. Coletando dois indicadores por UF

Vamos investigar: **População da UF e PIB per capita estão correlacionados?**

In [ ]:
df_pib = obter_pib_per_capita_uf()
df_pop = obter_populacao_uf(ano=2022)

dados = df_pib.merge(df_pop, on="uf")
dados = dados.dropna()
dados["log_pop"] = np.log10(dados["populacao"])  # log para reduzir assimetria
dados.head()

## 5. Correlações

In [ ]:
r_pearson,  p_pearson  = stats.pearsonr(dados["log_pop"], dados["pib_per_capita"])
r_spearman, p_spearman = stats.spearmanr(dados["populacao"], dados["pib_per_capita"])

print(f"Pearson  (log_pop vs pib_pc):  r = {r_pearson:+.3f}, p = {p_pearson:.4f}")
print(f"Spearman (pop vs pib_pc):      r = {r_spearman:+.3f}, p = {p_spearman:.4f}")

## 6. Scatter com regressão

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.regplot(data=dados, x="log_pop", y="pib_per_capita",
            scatter_kws={"s": 70, "alpha": 0.7, "color": "steelblue"},
            line_kws={"color": "darkred", "linewidth": 2}, ax=ax)
for _, row in dados.iterrows():
    ax.annotate(row["uf"][:2].upper(), (row["log_pop"], row["pib_per_capita"]),
                fontsize=8, alpha=0.7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("log10(População)")
ax.set_ylabel("PIB per capita (R$)")
ax.set_title(f"PIB per capita vs População (log) — r = {r_pearson:+.3f}",
             fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, "aula09_scatter_regressao.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Regressão Linear via `statsmodels`

`statsmodels` traz o relatório completo (coeficientes, p-valores, IC, $R^2$, diagnósticos).

In [ ]:
X = sm.add_constant(dados["log_pop"])   # adiciona intercepto
y = dados["pib_per_capita"]

modelo = sm.OLS(y, X).fit()
print(modelo.summary())

### Como ler o `summary`:
- **coef** → estimativa de $\hat\beta$.
- **std err** → erro-padrão do coeficiente.
- **t** e **P>|t|** → teste de $H_0$: $\beta = 0$.
- **R-squared** → fração da variância explicada.
- **F-statistic** → teste global do modelo.
- **Durbin-Watson** ≈ 2 indica resíduos não autocorrelacionados.

In [ ]:
# Acesso direto às quantidades de interesse
print(f"Intercepto β0:  {modelo.params.iloc[0]:,.2f}")
print(f"Inclinação β1:  {modelo.params.iloc[1]:,.2f}")
print(f"R² = {modelo.rsquared:.4f}")
print(f"R² ajustado = {modelo.rsquared_adj:.4f}")
print(f"F = {modelo.fvalue:.2f}, p = {modelo.f_pvalue:.4f}")

## 8. Diagnóstico de resíduos

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

residuos = modelo.resid
ajustados = modelo.fittedvalues

# 1. Resíduos vs Ajustados — deve não ter padrão
axes[0,0].scatter(ajustados, residuos, alpha=0.7, color="steelblue")
axes[0,0].axhline(0, color="red", linestyle="--")
axes[0,0].set_xlabel("Valores ajustados")
axes[0,0].set_ylabel("Resíduos")
axes[0,0].set_title("Resíduos vs Ajustados (linearidade/homocedasticidade)")

# 2. Q-Q plot dos resíduos
stats.probplot(residuos, dist="norm", plot=axes[0,1])
axes[0,1].set_title("Q-Q plot dos resíduos (normalidade)")
axes[0,1].get_lines()[0].set_color("steelblue")
axes[0,1].get_lines()[1].set_color("red")

# 3. Histograma dos resíduos
sns.histplot(residuos, kde=True, ax=axes[1,0], color="orange", edgecolor="white")
axes[1,0].set_title("Distribuição dos resíduos")
axes[1,0].set_xlabel("Resíduo")

# 4. Scale-Location (raiz dos resíduos absolutos padronizados)
resid_pad = residuos / residuos.std()
axes[1,1].scatter(ajustados, np.sqrt(np.abs(resid_pad)), alpha=0.7, color="purple")
axes[1,1].set_xlabel("Valores ajustados")
axes[1,1].set_ylabel("√|resíduo padronizado|")
axes[1,1].set_title("Scale-Location (homocedasticidade)")

plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, "aula09_diagnostico_residuos.png"), dpi=150, bbox_inches="tight")
plt.show()

## 9. Teste formal de homocedasticidade — Breusch-Pagan

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

bp = het_breuschpagan(residuos, X)
print(f"Breusch-Pagan: LM = {bp[0]:.4f}, p = {bp[1]:.4f}")
print("H0: homocedasticidade. Se p < 0,05 → variância NÃO é constante.")

## 10. Predição

In [ ]:
# Imagina uma UF com 5 milhões de habitantes (log10 = 6.7)
nova_pop_log = pd.DataFrame({"const": [1.0], "log_pop": [6.7]})
predicao = modelo.get_prediction(nova_pop_log).summary_frame(alpha=0.05)
print(f"Para log10(pop) = 6.7  → População ≈ 5 mi:")
print(f"PIB per capita previsto: R$ {predicao['mean'].iloc[0]:,.2f}")
print(f"IC 95% para a média:    [R$ {predicao['mean_ci_lower'].iloc[0]:,.2f}, "
      f"R$ {predicao['mean_ci_upper'].iloc[0]:,.2f}]")
print(f"IC 95% para predição:   [R$ {predicao['obs_ci_lower'].iloc[0]:,.2f}, "
      f"R$ {predicao['obs_ci_upper'].iloc[0]:,.2f}]")

## Exercícios

1. Calcule a correlação entre **PIB per capita** e **PIB total** (PIB × população).
2. Ajuste a regressão **sem o log** e compare $R^2$ com a versão log-linear.
3. Identifique os 3 pontos com maior resíduo (em valor absoluto). São os mesmos
   outliers que detectamos na aula 03?

**Próxima aula:** Análise integrada com Plotly interativo.